In [4]:
import numpy as np

def needleman_wunsch_affine(s1, s2, match=3, mismatch=-3, gap_open=-10, gap_extend=-1):
    m, n = len(s1), len(s2)
    INF = -100000.0
    D = np.full((m+1, n+1), INF)
    E = np.full((m+1, n+1), INF)  # deletion: ends with gap in s2
    F = np.full((m+1, n+1), INF)  # insertion: ends with gap in s1

    D[0][0] = 0

    for i in range(1, m+1):
        D[i][0] = gap_open + (i-1) * gap_extend
        E[i][0] = D[i][0]
        F[i][0] = INF

    for j in range(1, n+1):
        D[0][j] = gap_open + (j-1) * gap_extend
        F[0][j] = D[0][j]
        E[0][j] = INF

    for i in range(1, m+1):
        for j in range(1, n+1):
            s = match if s1[i-1] == s2[j-1] else mismatch
            E[i][j] = max(D[i-1][j] + gap_open, E[i-1][j] + gap_extend)
            F[i][j] = max(D[i][j-1] + gap_open, F[i][j-1] + gap_extend)
            D[i][j] = max(D[i-1][j-1] + s, E[i][j], F[i][j])

    score = D[m][n]

    # Traceback
    align1 = []
    align2 = []
    i, j = m, n
    while i > 0 or j > 0:
        if i == 0:
            align1.append('-')
            align2.append(s2[j-1])
            j -= 1
            continue
        if j == 0:
            align1.append(s1[i-1])
            align2.append('-')
            i -= 1
            continue
        s = match if s1[i-1] == s2[j-1] else mismatch
        if D[i][j] == D[i-1][j-1] + s:
            align1.append(s1[i-1])
            align2.append(s2[j-1])
            i -= 1
            j -= 1
        elif D[i][j] == E[i][j]:
            align1.append(s1[i-1])
            align2.append('-')
            i -= 1
        elif D[i][j] == F[i][j]:
            align1.append('-')
            align2.append(s2[j-1])
            j -= 1
        else:
            raise ValueError("Traceback error")

    align1.reverse()
    align2.reverse()
    alignment1 = ''.join(align1)
    alignment2 = ''.join(align2)

    return score, D, alignment1, alignment2

In [5]:
#Наши последовательности:
s1 = 'ATGCAGCAGCAGCCA'
s2 = 'ATATAT'

In [10]:
#Сначала запустим линейный штраф (Gap = -4):
#В случае моей реализации просто положим gap_open = gap_extend= -4

Res = needleman_wunsch_affine(s1, s2, match=3, gap_open=-4, gap_extend=-4)

print(f'Для модели с линейным штрафом:')
print(f'Score = {Res[0]}')
print(f'Мaтрица:')
print(Res[1])
print(f'Само выравнивание:')
print(Res[2])
print(Res[3])

Для модели с линейным штрафом:
Score = -30.0
Мaтрица:
[[  0.  -4.  -8. -12. -16. -20. -24.]
 [ -4.   3.  -1.  -5.  -9. -13. -17.]
 [ -8.  -1.   6.   2.  -2.  -6. -10.]
 [-12.  -5.   2.   3.  -1.  -5.  -9.]
 [-16.  -9.  -2.  -1.   0.  -4.  -8.]
 [-20. -13.  -6.   1.  -3.   3.  -1.]
 [-24. -17. -10.  -3.  -2.  -1.   0.]
 [-28. -21. -14.  -7.  -6.  -5.  -4.]
 [-32. -25. -18. -11. -10.  -3.  -7.]
 [-36. -29. -22. -15. -14.  -7.  -6.]
 [-40. -33. -26. -19. -18. -11. -10.]
 [-44. -37. -30. -23. -22. -15. -14.]
 [-48. -41. -34. -27. -26. -19. -18.]
 [-52. -45. -38. -31. -30. -23. -22.]
 [-56. -49. -42. -35. -34. -27. -26.]
 [-60. -53. -46. -39. -38. -31. -30.]]
Само выравнивание:
ATGCAGCAGCAGCCA
AT-----A-TA---T


In [11]:
#Теперь запустим Аффинный штраф (gap_open=-10, gap_extend=-1)

res = needleman_wunsch_affine(s1, s2, match=3, gap_open=-10, gap_extend=-1)

print(f'Для модели с аффинным штрафом:')
print(f'Score = {res[0]}')
print(f'Мaтрица:')
print(res[1])
print(f'Само выравнивание:')
print(res[2])
print(res[3])

Для модели с аффинным штрафом:
Score = -18.0
Мaтрица:
[[  0. -10. -11. -12. -13. -14. -15.]
 [-10.   3.  -7.  -8.  -9. -10. -11.]
 [-11.  -7.   6.  -4.  -5.  -6.  -7.]
 [-12.  -8.  -4.   3.  -7.  -8.  -9.]
 [-13.  -9.  -5.  -7.   0. -10. -11.]
 [-14. -10.  -6.  -2. -10.   3.  -7.]
 [-15. -11.  -7.  -9.  -5.  -7.   0.]
 [-16. -12.  -8. -10. -12.  -8. -10.]
 [-17. -13.  -9.  -5. -13.  -9. -11.]
 [-18. -14. -10. -12.  -8. -10. -12.]
 [-19. -15. -11. -13. -15. -11. -13.]
 [-20. -16. -12.  -8. -16. -12. -14.]
 [-21. -17. -13. -15. -11. -13. -15.]
 [-22. -18. -14. -16. -18. -14. -16.]
 [-23. -19. -15. -17. -19. -15. -17.]
 [-24. -20. -16. -12. -20. -16. -18.]]
Само выравнивание:
ATGCAGCAGCAGCCA
AT--------ATA-T


Аффинная модель лучше всего описывает биологическую особенность, при которой вставки и делеции в ДНК происходят целыми блоками, а не как одиночные независимые события. Сама модель, как мы видим, описывает ситуацию, при которой открытие гэпа требует больших ресурсов (как раз более высокий штраф), а его продолжение уже относительно дешево. Это позволяет моделировать такие события как единое целое. Если рассматривать примеры, то данные ситуации в геноме происходят например при залипании полимераз. 